# VolHack data overview

A first, read-only look at the supplied minute-return history and competition prediction timestamps. Run this notebook from the repository root so the explicit input paths resolve. It does not fit a model or write any derived data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.6g}".format

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "epfl-vol-hack/dataset.csv"
SUBMISSION_PATH = PROJECT_ROOT / "epfl-vol-hack/sample_submission.csv"
assert DATA_PATH.is_file(), f"Missing input: {DATA_PATH.resolve()}"
assert SUBMISSION_PATH.is_file(), f"Missing input: {SUBMISSION_PATH.resolve()}"

## Load and validate the supplied inputs

`timestamp` is converted only for presentation. Returns and submission IDs remain untouched.

In [ ]:
returns = pd.read_csv(DATA_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

assert returns.columns.tolist() == ["timestamp", "ret"]
assert submission.columns.tolist() == ["ID", "p_up"]
assert returns["timestamp"].is_monotonic_increasing
assert np.all(np.diff(returns["timestamp"].to_numpy()) == 60)
assert submission["ID"].is_unique

returns["datetime"] = pd.to_datetime(returns["timestamp"], unit="s", utc=True)
submission["datetime"] = pd.to_datetime(submission["ID"], unit="s", utc=True)
returns["observed"] = returns["ret"].notna()

overview = pd.Series(
    {
        "minute rows": len(returns),
        "date range (UTC)": (
            f"{returns.datetime.iloc[0]:%Y-%m-%d} to "
            f"{returns.datetime.iloc[-1]:%Y-%m-%d}"
        ),
        "observed returns": int(returns.observed.sum()),
        "missing returns": int((~returns.observed).sum()),
        "observed share": returns.observed.mean(),
        "submission IDs": len(submission),
        "ID spacing, median (minutes)": np.median(np.diff(submission.ID) / 60),
    },
    name="value",
)
overview.to_frame()

## Missing-return windows and target alignment

Competition cases are represented by contiguous missing-return windows. This cell measures the windows directly and locates every target ID within its window; it does not attempt to reconstruct any missing return.

In [ ]:
missing = ~returns.observed.to_numpy()
gap_starts = np.flatnonzero(missing & np.r_[True, ~missing[:-1]])
gap_ends = np.flatnonzero(missing & np.r_[~missing[1:], True])
gap_lengths = gap_ends - gap_starts + 1

first_timestamp = returns.timestamp.iloc[0]
target_rows = ((submission.ID.to_numpy() - first_timestamp) // 60).astype(int)
target_gap_index = np.searchsorted(gap_starts, target_rows, side="right") - 1
assert np.all(target_rows >= gap_starts[target_gap_index])
assert np.all(target_rows <= gap_ends[target_gap_index])

target_offset = target_rows - gap_starts[target_gap_index]
gap_summary = pd.DataFrame(
    {
        "gap count": [len(gap_lengths)],
        "typical competition-gap length (minutes)": [
            pd.Series(gap_lengths[gap_lengths > 1]).mode().iloc[0]
        ],
        "target offset from gap start (minutes)": [pd.Series(target_offset).mode().iloc[0]],
        "targets inside a missing window": [len(target_rows)],
    }
)
gap_summary

In [ ]:
example_target = target_rows[0]
start = example_target - 45
stop = example_target + 120
window = returns.iloc[start:stop].copy()

fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(
    window.datetime, window.observed.astype(int), s=10, color="tab:blue", label="return supplied"
)
ax.axvline(
    returns.datetime.iloc[example_target], color="tab:red", lw=2, label="submission target ID"
)
ax.set(
    title="Observation pattern around the first submission target",
    xlabel="UTC time",
    ylabel="Return supplied",
    ylim=(-0.1, 1.1),
)
ax.legend(frameon=True)
fig.autofmt_xdate()
plt.show()

## Return distribution

The left panel keeps the full scale for tail awareness. The right panel limits the view to the 0.5th–99.5th percentiles so the typical one-minute move is visible. The values themselves are unchanged.

In [ ]:
observed_returns = returns.loc[returns.observed, "ret"]
low, high = observed_returns.quantile([0.005, 0.995])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(observed_returns, bins=300, color="tab:blue", alpha=0.85)
axes[0].set(title="All observed one-minute returns", xlabel="Return", ylabel="Minutes")
axes[1].hist(observed_returns.clip(low, high), bins=200, color="tab:blue", alpha=0.85)
axes[1].set(
    title="Central 99% of returns (winsorized for display)",
    xlabel="Return",
    ylabel="Minutes",
)
fig.tight_layout()
plt.show()

observed_returns.describe(percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]).to_frame("ret")

## UTC intraday profile

This compares data availability and realised one-minute volatility by UTC hour. It is descriptive only; any future time-of-day feature must be built from information available at the prediction timestamp.

In [ ]:
hourly = (
    returns.assign(utc_hour=returns.datetime.dt.hour)
    .groupby("utc_hour", observed=True)
    .agg(observed_share=("observed", "mean"), return_std=("ret", "std"), count=("ret", "count"))
)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(hourly.index, hourly.observed_share, marker="o", color="tab:blue")
axes[0].set(
    title="Return availability by UTC hour", ylabel="Observed share", ylim=(0, 1)
)
axes[1].plot(hourly.index, hourly.return_std, marker="o", color="tab:orange")
axes[1].set(
    title="Observed-return volatility by UTC hour",
    xlabel="UTC hour",
    ylabel="Standard deviation",
)
axes[1].set_xticks(range(24))
fig.tight_layout()
plt.show()

hourly

## Initial takeaways

- The timestamps form an uninterrupted minute grid; missingness is in `ret`, not in the clock.
- Every supplied target aligns to the same point in a 106-minute missing-return window, so gap boundaries are a useful audit constraint.
- The distribution plot distinguishes the dense typical range from rare extreme moves; later log-loss models should be calibrated conservatively around these tails.
- The intraday chart is a prompt to test time-of-day features under chronological, leakage-safe validation rather than evidence of predictive value by itself.